In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
import transformers
from datetime import datetime
from pathlib import Path

# print(torch.__version__)
# print(transformers.__version__)

In [ ]:
last_modified = datetime.fromtimestamp(
    Path("examuse_hw.ipynb").stat().st_mtime
)

In [5]:
last_modified

datetime.datetime(2026, 1, 28, 15, 25, 41, 174258)

*Last updated:* {{ last_modified.strftime("%Y-%m-%d") }}

*Created:* {{< meta date >}}  
*Last updated:* {{ last_modified.strftime("%Y-%m-%d") }}


# Stage 1. Paper Summary

## Title, authors

- **Title**: Practical and Reproducible Symbolic Music Generation by Large Language Models with Structural Embeddings
- **Authors**: Seungyeon Rhyu, Kichang Yang, Sungjun Cho, Jaehyeon Kim, Kyogu Lee, Moontae Lee
- [ArXiv preprint](https://arxiv.org/abs/2407.19900)
- Submitted on 29 Jul 2024

## Description

Large language models should be naturally extended to generate symbolic music. However, some music-specific aspects turn out to be challenging for those models. In particular, MIDI data often lack annotations of bars and beats, a fact that impedes effective tokenization. Heuristic algorithms trying to generate such annotations cannot be applied reliably because in most of the MIDI data generated from real music audio, performers start or end notes at points deviating from the scripted tempo. 

Music-generation frameworks like MuseNet [@huangMusicTransformer2018] introduced additional *structural embeddings* to extract the structural context of music from MIDI files. However, according to Rhyu et al. [-@rhyuPracticalReproducibleSymbolic2024], their choices were not well-argumented and the experiments they made were not well-documented. These limitations hinder further testing, and potential application to data that lacks domain annotations. 

To address these limitations, Rhyu et al.:

- train a vanilla GPT-2 model
- implement the structural embeddings proposed by MuseNet
- perform ablation studies for different initialization methods

The overarching aim is two fold:

1. Gain insights into how to effectively encode structural information without manual annotations
2. Provide an accessible pipeline to train models on large music datasets. 

## Summary of the main ideas / achievements / challenges / future work (if any). Notable results (if any)

**Big research question** 
- How to effectively toekenize MIDI data that often lack annotations of bars and beats

**Smaller research question**, directly addressed by Rhyu et al. 
- Implement reproducible structural embeddings.


**Achievements**
- Reproducible structural tokenization
- Online demonstration of the generated music

**Notable Results**
- Structural embeddings are beneficial for symbolic music generation

**Challenges and future work**
- Better objective metrics are needed to capture the structureness of data beyond simple repetition. 


### Equations / formulas

- Training uses next-token prediction task. 
	- At each step, predict $x_i$ from $x_{<i}$ by optimizing 
    $J = -\sum{\log p(x_i | x_{<i})} $
    This loss fucntion is cross-entropy. 

- Sinusiodal Initialization, following Vaswani et al. [-@vaswani2017AttentionAllYou] and Guo et al. [-@guo2023DomainknowledgeinspiredMusicEmbedding]
    - $SE_{(k, 2i)} = \sin (k / (10000 / w)^{2i/d}) $
    - $SE_{(k, 2i+1)} = \cos (k / (10000 / w)^{2i/d}) $


### Terms, concepts

- **Structural Embeddings** - vector representations that explicitly capture the structural relationships or hierarchical organization of data points. Rhyu et al. use 4 types of structural embeddings:
    - Part (out of 128 subdivisions of each song)
    - Type of token
    - Time
    - Pitch Class

- **Initialization Methods**
1. truncated normal initialization
2. sinusoidal initialization, applied only to the time-related embeddings 

- Metrics for objective evaluation of musical quality

- **Objective evaluation** of musical quality 
	- _Structureness indicator_ (SI): measures the largest degree of repeatedness among various intervals in generated music; a fitness score based on a similarity matrix
	- _Chord Progression Validation Rationality_ (CPVR): conditional probability of the existence of each unique chord n-gram given its previous n-gram within the generated music.
	- _Chord Progression Irregularity_ (CPI): measures the ratio of unique chord n-grams from generated music, compensating CPVR that yields high scores with frequently occurring chords

- **Subjective Evaluation** of music
	- A-vs-B human-rating procedure. 
	- Four human raters
	- Rating scales:
		- Naturalness
		- Prompt maintenance

- **Fitness Scape Plots**
...

## Stage 2: Architecture and key result (usually figure / table) 

## Architecture

- Backbone architecture: GPT-2
- No sparse attention and mixup 
- Concatenate structural embeddings with input token embeddings
- Positional encoding to the resulting embeddings 

### Training

- uses next-token prediction task. 
- At each step, predict $x_i$ from $x_{<i}$ by optimizing <paste loss fn> 


The architecture & training procedure are illustrated in Figure 3, left panel: 


![Figure 3, left](Fig3_left.png)

### Inference: 

- the model autoregressively generates the sequence given a prompt of MIDI tokens
- after each token is generated, its four structural labels are extracted using rule-based modules

Illustrated in Figure 3, right panel: 

![Figure 3, right](Fig3_right.png)

## Key Result

- Adding structural embeddings to GPT2 enhances the ability of the model to capture structural aspects of music
- Random initialization genarates music that sounds better to the human ear
- Sinusoidal initialization of temporal embeddings produces more repeated patterns and common chords 


Scape Plot [@muller2012ScapePlotRepresentation], see also [this online notebook](https://www.audiolabs-erlangen.de/resources/MIR/FMP/C4/C4S3_ScapePlot.html)

## Stage 3: Reproduction and ideas

### Reproduction

#### Setup

#### Results

### Ideas for extensions

#### Limitations

#### Jazz/BebopNet angle

# References


In [2]:
import numpy as np

correct_probs = np.array([0.9, 0.6, 0.33, 0.1])
loss = -np.log(correct_probs)

for p, l in zip(correct_probs, loss):
    print(f"Predicted probability for true class = {p:.2f} → CE loss = {l:.3f}")

Predicted probability for true class = 0.90 → CE loss = 0.105
Predicted probability for true class = 0.60 → CE loss = 0.511
Predicted probability for true class = 0.33 → CE loss = 1.109
Predicted probability for true class = 0.10 → CE loss = 2.303


In [6]:
np.log(.9)

np.float64(-0.10536051565782628)

In [7]:
import numpy as np

logits = np.array([2.0, 1.0, 0.1])
exp_vals = np.exp(logits)
softmax = exp_vals / np.sum(exp_vals)

print("Logits:", logits)
print("Softmax probabilities:", softmax)

Logits: [2.  1.  0.1]
Softmax probabilities: [0.65900114 0.24243297 0.09856589]
